# 04 — Đo tốc độ Qwen trên acc 0 trong lúc acc 1/2 vẫn chạy

Kaggle → Add Input thư viện cũ **vmb-full-inputs-d6ab9a432ac6-1792733353195313801**
→ Internet ON → T4 x2 → Save & Run All. Không cần kế hoạch GPU mới,
không nhập/gộp/chạy lại task cũ. Không dùng GPU acc 1/2 đang chạy.

Hai GPU đo riêng native batch 16, nhóm theo token và nhánh FP16 math SDPA
nếu runtime hỗ trợ. Mọi lựa chọn phải nhanh hơn ít nhất 5%, vượt cosine
>=0,99999 và sai số tuyệt đối <=0,001, rồi xác nhận bằng phép đo thứ hai.
Nhánh nhanh không đạt sẽ bị loại. Weights NF4, text, tokenizer và pooling
gốc giữ nguyên; lựa chọn attention trung gian được ghi rõ trong báo cáo.

Notebook đọc tối đa ba task slice từ thư viện text đã có, không đọc Drive,
vector cũ hoặc toàn bộ corpus. Tải model/runtime vẫn cần thiết khi phiên
mới. Không ghi task vào hàng đợi đã hoàn thành. Outputs chỉ có hai JSON
đo tốc độ và log; gửi các dòng QWEN_BENCHMARK_GPU để chốt cấu hình.

In [ ]:
#@title 1. Kaggle T4 x2 — dùng thư viện text cũ, không cần kế hoạch mới
import hashlib,importlib,json,os,re,subprocess,sys,time
from pathlib import Path
if not Path('/kaggle/input').is_dir() or Path('/var/colab/hostname').is_file():
    raise RuntimeError('Notebook đo tốc độ này chạy trên Kaggle T4 x2.')
KAGGLE_ACCOUNT_ID=0
libraries={}
for path in Path('/kaggle/input').rglob('job.json'):
    item=json.loads(path.read_text())
    if item.get('schema')=='full-data-text-library-v1':
        libraries[item.get('manifest_sha256')]=(path.parent,item)
if len(libraries)!=1:
    raise RuntimeError('Add Input đúng một thư viện text vmb-full-inputs-d6ab9a432ac6; không cần xuất dataset mới.')
JOB_DIR,JOB=next(iter(libraries.values()))
body={k:v for k,v in JOB.items() if k!='manifest_sha256'}
sha=hashlib.sha256(json.dumps(body,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
if (sha!=JOB['manifest_sha256'] or sha!='d6ab9a432ac64e5aadb39a826b7dc2176094cc66d4f4e3d9e17c121e071c0711'
        or JOB.get('repo_url')!='https://github.com/Platypus27-coder/VietMedBridge.git'
        or not re.fullmatch('[0-9a-f]{40}',JOB.get('code_commit',''))):
    raise ValueError('Thư viện hoặc producer không khớp gói đã freeze.')
print('QWEN_BENCHMARK_INPUT_READY: thư viện cũ; tối đa ba task slice, không ghi task hoàn thành.',flush=True)

In [ ]:
# GPU setup: run only after reviewing the input inventory.
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)
CHECKOUT = Path('/kaggle/temp/VietMedBridge-new-data')
WORK_DIR = Path('/kaggle/temp/vmb_new_data')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(WORK_DIR/'hf_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if not (CHECKOUT/'.git').is_dir():
    subprocess.run(['git','clone','--no-checkout',JOB['repo_url'],str(CHECKOUT)], check=True)
if subprocess.check_output(['git','-C',str(CHECKOUT),'remote','get-url','origin'], text=True).strip() != JOB['repo_url']:
    raise ValueError('Checkout belongs to another repository.')
subprocess.run(['git','-C',str(CHECKOUT),'fetch','--depth','1','origin',JOB['code_commit']], check=True)
subprocess.run(['git','-C',str(CHECKOUT),'checkout','--detach','FETCH_HEAD'], check=True)
print('Installing pinned encoder dependencies; no Google Drive mount on Kaggle.', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.[strong]'], cwd=CHECKOUT, check=True)
sys.path[:0] = [str(JOB_DIR),str(CHECKOUT/'src')]
importlib.invalidate_caches()
from vietmedbridge.portable_embeddings import runtime_install_commands
for command in runtime_install_commands(JOB['runtime']):
    subprocess.run(command, check=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(CHECKOUT/'src')+os.pathsep+str(JOB_DIR)
probe = "import torch; print('FRESH CUDA:',torch.__version__,torch.cuda.is_available(),torch.cuda.device_count(),flush=True); assert torch.cuda.is_available() and torch.cuda.device_count()==2"
subprocess.run([sys.executable,'-u','-c',probe], env=env, check=True)
print('READY | batch',JOB['batch'],'| team',JOB['team'],'| account',KAGGLE_ACCOUNT_ID,flush=True)
(WORK_DIR/'qwen_token_batching.py').write_text('"""Measured execution-only batching around the immutable corpus encoder.\n\nNo weights, tokenizer options, text, pooling, vector dimensions or cache identity\nare changed. The native forward is used, with a bounded on-device canary before\nselecting any faster batching order. There is no local model inference here.\n"""\nfrom contextlib import contextmanager\nimport gc\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom vietmedbridge.artifacts import atomic_json\nfrom vietmedbridge.embeddings import validate_vectors\n\n\nPOLICY = \'qwen-token-batches-v2\'\n\n\ndef token_groups(lengths, cap, budget):\n    order = sorted(range(len(lengths)), key=lambda i: (lengths[i], i))\n    group = []\n    for i in order:\n        if group and (len(group) >= cap or (len(group)+1)*lengths[i] > budget):\n            yield group\n            group = []\n        group.append(i)\n    if group:\n        yield group\n\n\ndef vector_agreement(reference, candidate):\n    if reference.shape != candidate.shape or not np.isfinite(candidate).all():\n        return False, {\'reason\':\'shape_or_finite\'}\n    cosine = np.sum(reference*candidate, axis=1) / np.maximum(\n        np.linalg.norm(reference,axis=1)*np.linalg.norm(candidate,axis=1), 1e-12)\n    minimum = float(cosine.min())\n    maximum = float(np.max(np.abs(reference-candidate)))\n    return minimum >= 0.99999 and maximum <= 0.001, {\'min_cosine\':minimum,\'max_abs_error\':maximum}\n\n\nclass TokenBatchEncoder:\n    def __init__(self, native, report_path, *, deadline=None, clock=time.perf_counter, report=print,\n                 attention_tuning=False):\n        self.native = native\n        self.encoder = getattr(native,\'encoder\',native)\n        self.identity, self.dimension = native.identity, native.dimension\n        self.path = Path(report_path)\n        self.deadline, self.clock, self.report = deadline, clock, report\n        self.selected, self.safe_batch_size = None, 16\n        self.selected_kernel,self.selected_budget = \'native\',24_576\n        self.attention_tuning = attention_tuning\n        self.caps, self.ooms = {}, 0\n        self.attempted = False\n\n    def _attention_api(self):\n        backend=getattr(getattr(self.encoder.torch,\'backends\',None),\'cuda\',None)\n        setter=getattr(backend,\'allow_fp16_bf16_reduction_math_sdp\',None)\n        getter=getattr(backend,\'fp16_bf16_reduction_math_sdp_allowed\',None)\n        return (setter,getter) if callable(setter) and callable(getter) else (None,None)\n\n    @contextmanager\n    def _attention(self,kernel):\n        if kernel==\'native\':\n            yield\n            return\n        setter,getter=self._attention_api()\n        if kernel!=\'fp16_math\' or setter is None:\n            raise ValueError(\'Unsupported attention execution policy.\')\n        previous=getter()\n        try:\n            setter(True)\n            yield\n        finally:\n            setter(previous)\n\n    def lengths(self, texts):\n        # Exactly the native tokenizer\'s options, without batch padding.\n        ids = self.encoder.tokenizer(list(texts),padding=False,truncation=False)[\'input_ids\']\n        lengths = [len(row) for row in ids]\n        if any(n > self.encoder.spec[\'max_length\'] for n in lengths):\n            raise ValueError(\'Qwen input exceeds the frozen max_length; no text was truncated.\')\n        return lengths\n\n    def _forward(self, texts, indices, lengths, kernel):\n        bucket = max(lengths[i] for i in indices).bit_length()\n        cap = self.caps.get((kernel,bucket),len(indices))\n        if len(indices) > cap:\n            return np.concatenate([self._forward(texts,indices[i:i+cap],lengths,kernel)\n                                   for i in range(0,len(indices),cap)],axis=0)\n        try:\n            self.encoder.input_role = \'corpus\'\n            return np.asarray(self.encoder._batch([texts[i] for i in indices]),dtype=np.float32)\n        except self.encoder.torch.OutOfMemoryError:\n            if len(indices) == 1:\n                raise\n            self.ooms += 1\n        # Drop the exception/traceback before freeing CUDA allocations.\n        cap = max(1,len(indices)//2)\n        self.caps[kernel,bucket] = cap\n        gc.collect()\n        self.encoder.torch.cuda.empty_cache()\n        self.report(f\'QWEN_BATCH_OOM: token bucket {bucket}; retry same texts with batch {cap}\',flush=True)\n        return np.concatenate([self._forward(texts,indices[i:i+cap],lengths,kernel)\n                               for i in range(0,len(indices),cap)],axis=0)\n\n    def _run(self, texts, lengths, cap, kernel=\'native\', budget=24_576):\n        result = np.empty((len(texts),self.dimension),np.float32)\n        with self._attention(kernel):\n            for indices in token_groups(lengths,cap,budget):\n                values = self._forward(texts,indices,lengths,kernel)\n                validate_vectors(values,len(indices),self.dimension)\n                result[indices] = values\n        return result\n\n    def _calibrate(self, texts, lengths):\n        self.attempted = True\n        started = self.clock()\n        order = sorted(range(len(texts)),key=lambda i:lengths[i])\n        indices = sorted({order[int(i)] for i in np.linspace(0,len(order)-1,min(64,len(order)))})\n        sample = [texts[i] for i in indices]\n        sample_lengths = [lengths[i] for i in indices]\n        self.report(f\'QWEN_TUNING: {len(sample)} existing task texts; native batch 16 versus token batches 16/32/64\',flush=True)\n        # Warm up only existing task texts. These vectors are never published.\n        self.native.encode(sample[:4],batch_size=16)\n        tick = self.clock()\n        reference = np.asarray(self.native.encode(sample,batch_size=16),dtype=np.float32)\n        baseline = max(self.clock()-tick,1e-9)\n        validate_vectors(reference,len(sample),self.dimension)\n        best, trials = baseline, []\n        candidates=[(16,\'native\',24_576),(32,\'native\',24_576),(64,\'native\',24_576)]\n        if self.attention_tuning:\n            candidates.append((64,\'native\',49_152))\n            setter,getter=self._attention_api()\n            if setter is not None and not getter():\n                candidates.insert(1,(32,\'fp16_math\',24_576))\n                candidates.insert(2,(64,\'fp16_math\',49_152))\n        for cap,kernel,budget in candidates:\n            if self.clock()-started > (120 if self.attention_tuning else 90) or (self.deadline is not None and time.monotonic()+180 >= self.deadline):\n                break\n            tick = self.clock()\n            # Include token-length preparation in the candidate measurement.\n            self.lengths(sample)\n            try:\n                values = self._run(sample,sample_lengths,cap,kernel,budget)\n            except ValueError:\n                trials.append({\'batch_cap\':cap,\'kernel\':kernel,\'compatible\':False,\'reason\':\'vector_validation_failed\'})\n                continue\n            seconds = max(self.clock()-tick,1e-9)\n            compatible, agreement = vector_agreement(reference,values)\n            trial = {\'batch_cap\':cap,\'kernel\':kernel,\'max_padded_tokens\':budget,\n                     \'seconds\':seconds,\'compatible\':compatible,**agreement}\n            trials.append(trial)\n            self.report(f\'QWEN_TUNING_TRIAL: {kernel} cap {cap} | {seconds:.2f}s | compatible {compatible} | native {baseline:.2f}s\',flush=True)\n            if compatible and seconds < best and seconds <= baseline/1.05:\n                best, self.selected = seconds, cap\n                self.selected_kernel,self.selected_budget=kernel,budget\n        confirmation=None\n        if self.selected is not None and self.attention_tuning:\n            # A second comparison prevents choosing a single noisy timing win.\n            tick=self.clock()\n            ref2=np.asarray(self.native.encode(sample,batch_size=16),dtype=np.float32)\n            baseline2=max(self.clock()-tick,1e-9)\n            tick=self.clock()\n            self.lengths(sample)\n            try:\n                values=self._run(sample,sample_lengths,self.selected,self.selected_kernel,self.selected_budget)\n                seconds2=max(self.clock()-tick,1e-9)\n                compatible,agreement=vector_agreement(ref2,values)\n            except ValueError:\n                compatible,agreement,seconds2=False,{\'reason\':\'vector_validation_failed\'},baseline2\n            confirmation={\'compatible\':compatible,\'native_seconds\':baseline2,\'selected_seconds\':seconds2,**agreement}\n            if not compatible or seconds2>baseline2/1.05:\n                self.selected,self.selected_kernel,self.selected_budget=None,\'native\',24_576\n                best=baseline\n            else:\n                baseline,best=baseline2,seconds2\n        self.safe_batch_size = self.selected or 16\n        record = {\'policy\':POLICY,\'encoder\':self.identity,\'mode\':\'token_batches\' if self.selected else \'native\',\n            \'sample_texts\':len(sample),\'baseline_seconds\':baseline,\'selected_seconds\':best,\n            \'batch_cap\':self.safe_batch_size,\'kernel\':self.selected_kernel,\n            \'max_padded_tokens\':self.selected_budget,\'trials\':trials,\'confirmation\':confirmation,\n            \'sample_tokens\':sum(sample_lengths),\'sample_token_range\':[min(sample_lengths),max(sample_lengths)],\n            \'measured_sample_speedup\':baseline/best,\'oom_backoffs\':self.ooms,\n            \'baseline_texts_per_second\':len(sample)/baseline,\'selected_texts_per_second\':len(sample)/best,\n            \'scope\':\'Startup canary, not a full-corpus quality or throughput certification.\'}\n        atomic_json(self.path,record)\n        self.report(f\'QWEN_TUNING_SELECTED: {record["mode"]} {self.selected_kernel} | cap {self.safe_batch_size} | measured sample speedup {baseline/best:.2f}x | {len(sample)/best:.2f} texts/s | report {self.path}\',flush=True)\n\n    def encode(self, texts, *, batch_size=16):\n        if not texts:\n            return np.empty((0,self.dimension),np.float32)\n        if not self.attempted and len(texts) >= 32:\n            self._calibrate(texts,self.lengths(texts))\n        if self.selected is None:\n            return self.native.encode(texts,batch_size=16)\n        lengths=self.lengths(texts)\n        try:\n            return self._run(texts,lengths,self.selected,self.selected_kernel,self.selected_budget)\n        except ValueError:\n            # Only the current uncommitted task is retried if later inputs expose\n            # a numerical issue; previous vector blocks remain unchanged.\n            self.selected,self.selected_kernel,self.safe_batch_size=None,\'native\',16\n            from vietmedbridge.artifacts import read_json\n            if self.path.is_file():\n                saved=read_json(self.path)\n                saved.update(active_mode=\'native\',runtime_fallback=\'vector_validation_failed\')\n                atomic_json(self.path,saved)\n            self.report(\'QWEN_TUNING_FALLBACK: native batch 16 for current uncommitted texts; previous checkpoints retained.\',flush=True)\n            return self.native.encode(texts,batch_size=16)\n\n\nclass TokenBatchOwner:\n    def __init__(self, owner, report_path, *, deadline=None, attention_tuning=False):\n        self.owner, self.path, self.deadline = owner, report_path, deadline\n        self.attention_tuning=attention_tuning\n\n    def for_role(self, role):\n        native = self.owner.for_role(role)\n        if role != \'corpus\' or native.identity.get(\'model_id\') != \'Qwen/Qwen3-Embedding-8B\':\n            return native\n        return TokenBatchEncoder(native,self.path,deadline=self.deadline,attention_tuning=self.attention_tuning)\n\n    def close(self):\n        self.owner.close()\n\n\ndef benchmark_texts(directory,library):\n    """Sample three bounded Qwen task slices; never walk corpus/vector files."""\n    import pyarrow.parquet as pq\n    from vietmedbridge.artifacts import digest_json,verify_file\n    from vietmedbridge.content_embeddings import key\n    tasks=[t for t in library[\'tasks\'] if t[\'family\']==\'qwen\']\n    if not tasks:\n        raise ValueError(\'Library has no Qwen task to benchmark.\')\n    selected=[tasks[i] for i in sorted({0,len(tasks)//2,len(tasks)-1})]\n    values=[]\n    for task in selected:\n        if Path(task[\'asset\']).name!=task[\'asset\']:\n            raise ValueError(\'Unsafe benchmark text asset path.\')\n        path=Path(directory)/task[\'asset\']\n        verify_file(path,task[\'sha256\'])\n        table=pq.read_table(path,columns=[\'id\',\'text\'])\n        if len(table)!=library[\'assets\'][task[\'asset\']][\'rows\']:\n            raise ValueError(\'Benchmark asset row count changed.\')\n        rows=table.slice(task[\'offset\'],task[\'rows\']).to_pylist()\n        if (len(rows)!=task[\'rows\'] or digest_json(sorted(r[\'id\'] for r in rows))!=task[\'keys_digest\']\n                or any(r[\'id\']!=key(r[\'text\']) for r in rows)):\n            raise ValueError(\'Benchmark task text/key binding changed.\')\n        indices=sorted({int(i) for i in np.linspace(0,len(rows)-1,min(24,len(rows)))})\n        values.extend(rows[i][\'text\'] for i in indices)\n    return values\n\n\ndef benchmark(directory,checkout,output,worker_id):\n    import torch\n    from full_queue_embeddings import validate_library\n    from vietmedbridge.artifacts import read_json\n    from vietmedbridge.portable_embeddings import embedding_runtime\n    from vietmedbridge.qwen_models import TorchQwenEncoder\n    from vietmedbridge.qwen_execution import configure_embedding_execution\n    job=validate_library(directory,checkout=checkout)\n    if not torch.cuda.is_available() or embedding_runtime()!=job[\'runtime\']:\n        raise RuntimeError(\'Benchmark must use the original producer CUDA runtime.\')\n    texts=benchmark_texts(directory,job)\n    owner=TorchQwenEncoder(job[\'config\'][\'second_dense\'])\n    try:\n        native=owner.for_role(\'corpus\')\n        expected=job[\'models\'][\'qwen\']\n        if native.identity!=expected[\'encoder\'] or native.dimension!=expected[\'dimension\']:\n            raise ValueError(\'Benchmark encoder differs from the existing vector producer.\')\n        configure_embedding_execution(native)\n        output=Path(output)\n        path=output/f\'worker-{worker_id}.json\'\n        tuned=TokenBatchEncoder(native,path,attention_tuning=True,deadline=time.monotonic()+900)\n        result=tuned.encode(texts,batch_size=64)\n        validate_vectors(result,len(texts),native.dimension)\n        report=read_json(path) if path.is_file() else {\'mode\':\'native\',\'reason\':\'not_enough_sample_texts\'}\n        operators=[]\n        try:\n            with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU]) as profile:\n                native.encode([texts[0],texts[-1]],batch_size=2)\n            operators=sorted({item.key for item in profile.key_averages() if \'scaled_dot_product\' in item.key})\n        except RuntimeError:\n            operators=[\'profiler_unavailable\']\n        report.update(state=\'BENCHMARK_COMPLETE_NO_TASKS_COMMITTED\',job=job[\'manifest_sha256\'],\n            gpu=torch.cuda.get_device_name(0),worker_id=worker_id,validated_texts=len(texts),\n            tasks_committed=0,native_sdpa_operators=operators)\n        atomic_json(path,report)\n        import json\n        print(\'QWEN_BENCHMARK_RESULT:\',json.dumps({k:v for k,v in report.items() if k!=\'encoder\'},ensure_ascii=False),flush=True)\n        return report\n    finally:\n        owner.close()\n\n\nif __name__==\'__main__\':\n    import argparse\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--job\',required=True)\n    parser.add_argument(\'--checkout\',required=True)\n    parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--worker-id\',type=int,required=True)\n    args=parser.parse_args()\n    benchmark(args.job,args.checkout,args.output,args.worker_id)\n',encoding='utf-8')
env['PYTHONPATH']=os.pathsep.join((str(WORK_DIR),str(CHECKOUT/'src'),str(JOB_DIR)))

In [ ]:
#@title 3. Đo độc lập hai GPU; kết quả nhỏ trong Outputs
output=Path('/kaggle/working/vmb_qwen_benchmark')
output.mkdir(parents=True,exist_ok=True)
processes=[]
started=time.monotonic()
try:
    for worker in (0,1):
        child=env.copy()
        child['CUDA_VISIBLE_DEVICES']=str(worker)
        command=[sys.executable,'-u',str(WORK_DIR/'qwen_token_batching.py'),
            '--job',str(JOB_DIR),'--checkout',str(CHECKOUT),'--output',str(output),
            '--worker-id',str(worker)]
        processes.append(subprocess.Popen(command,env=child))
    while any(p.poll() is None for p in processes):
        if any(p.poll() not in (None,0) for p in processes):
            raise RuntimeError('Benchmark worker lỗi; xem traceback, giữ model cache trong runtime.')
        if time.monotonic()-started>2400:
            raise RuntimeError('Benchmark vượt 40 phút; kiểm tra tải model/runtime trước khi chạy corpus.')
        print('QWEN_BENCHMARK_HEARTBEAT:',int(time.monotonic()-started),'s',flush=True)
        time.sleep(15)
    if any(p.returncode!=0 for p in processes):
        raise RuntimeError('Benchmark chưa hoàn tất trên cả hai GPU.')
finally:
    for p in processes:
        if p.poll() is None: p.terminate()
    for p in processes:
        try: p.wait(timeout=30)
        except subprocess.TimeoutExpired:
            p.kill();p.wait(timeout=10)
for worker in (0,1):
    report=json.loads((output/f'worker-{worker}.json').read_text())
    print('QWEN_BENCHMARK_GPU:',worker,'| native texts/s',report.get('baseline_texts_per_second'),
        '| selected texts/s',report.get('selected_texts_per_second'),
        '| sample speedup',report.get('measured_sample_speedup'),'| kernel',report.get('kernel'),flush=True)
print('BENCHMARK_COMPLETE_NO_TASKS_COMMITTED: chỉ đo tốc độ, chưa encode hàng đợi và không đổi checkpoint Drive.',flush=True)